# Try out Amazon Bedrock
In this short tutorial you prompt an LLM model on AWS Bedrock.

It includes different example use cases for Travel & Hospitality using Amazon Bedrock.

These sample use cases involve different tasks and prompt engeering techniques, as follows:

1. **Generate recommendations based on metadata**
- **Task**: Text Generation
- **Prompt Technique**: Zero-shot
2. **Estimate capacity for airlines or hotel properties based on historical data**
- **Task**: Complex Reasoning
- **Prompt Technique**: Chain-of-Thoughts (CoT)
3. **Create a question answering assistant for customer service**
- **Task**: Question Answering with Dialogue Asistant (without memory)
- **Prompt Technique**: Few-shot
4. **Summarize and classify content from media files transcription**
- **Task**: Text Summarization & Text Classification
- **Prompt Technique**: Zero-shot
5. **Create splash pages describing upcoming promotions**
- **Task**: Code Generation
- **Prompt Technique**: Zero-shot

Let's start by ensuring the Bedrock SDK is properly installed.

We'll also install a few libraries required in the notebook.

Let's start by ensuring the Bedrock SDK is properly installed.

We'll also install a few libraries required in the notebook.

In [1]:
! pip install --upgrade boto3  # 1.36.21
! pip install --upgrade botocore


Now we can import the libraries and setup the Bedrock client.

In [2]:
import json
from datetime import datetime

import boto3
import botocore

bedrock = boto3.client(service_name="bedrock", region_name="us-east-1")
bedrock_runtime = boto3.client('bedrock-runtime')

Let's get the list of Foundational Models supported in Bedrock at this time.

In [3]:
bedrock.list_foundation_models()

{'ResponseMetadata': {'RequestId': '43ced915-0db2-445d-81dc-02b60648fa2f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 28 Dec 2025 06:17:22 GMT',
   'content-type': 'application/json',
   'content-length': '133062',
   'connection': 'keep-alive',
   'x-amzn-requestid': '43ced915-0db2-445d-81dc-02b60648fa2f'},
  'RetryAttempts': 0},
 'modelSummaries': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/nvidia.nemotron-nano-12b-v2',
   'modelId': 'nvidia.nemotron-nano-12b-v2',
   'modelName': 'NVIDIA Nemotron Nano 12B v2 VL BF16',
   'providerName': 'NVIDIA',
   'inputModalities': ['TEXT', 'IMAGE'],
   'outputModalities': ['TEXT'],
   'responseStreamingSupported': True,
   'customizationsSupported': [],
   'inferenceTypesSupported': ['ON_DEMAND'],
   'modelLifecycle': {'status': 'ACTIVE'}},
  {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-sonnet-4-20250514-v1:0',
   'modelId': 'anthropic.claude-sonnet-4-20250514-v1:0',
   'modelName': 'Clau

# Warm up Exercise: Write a Poem about machine learning in the style of shakespear

In [18]:
# To do:

We will define an utility function for calling Bedrock.

This will help passing the proper body depending on the model invoked, and will store the results in a CSV file as well.

In [4]:
def call_bedrock(modelId, prompt_data):
    # Decide whether to use the Converse API (recommended for inference profiles and Meta Llama 3)
    is_inference_profile = isinstance(modelId, str) and modelId.startswith("arn:aws:bedrock:") and ":inference-profile/" in modelId
    is_meta = isinstance(modelId, str) and ("meta" in modelId)

    accept = 'application/json'
    contentType = 'application/json'

    # Use Converse for inference profiles or Meta models
    if is_inference_profile or is_meta:
        before = datetime.now()
        response = bedrock_runtime.converse(
            modelId=modelId,
            messages=[
                {
                    "role": "user",
                    "content": [{"text": prompt_data}]
                }
            ],
            inferenceConfig={
                "maxTokens": 4096,
                "temperature": 0,
                "topP": 0.9
            },
        )
        latency = (datetime.now() - before)

        # Extract text from Converse response
        output = response.get('output', {})
        message = output.get('message', {})
        content = message.get('content', [])
        response_text = None
        for part in content:
            if isinstance(part, dict) and 'text' in part:
                val = part.get('text')
                if isinstance(val, str):
                    response_text = val
                elif isinstance(val, dict):
                    # Some SDKs return structured text blocks
                    response_text = val.get('content') or val.get('text')
                break
        return response_text, latency

    # Legacy invoke_model paths for model IDs (non-ARN)
    if 'amazon' in modelId:
        body = json.dumps({
            "inputText": prompt_data,
            "textGenerationConfig": {
                "maxTokenCount": 4096,
                "stopSequences": [],
                "temperature": 0,
                "topP": 0.9
            }
        })
        before = datetime.now()
        response = bedrock_runtime.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)
        latency = (datetime.now() - before)
        response_body = json.loads(response.get('body').read())
        response_text = response_body.get('results', [{}])[0].get('outputText')
        return response_text, latency
    elif 'anthropic' in modelId:
        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 4096,
            "temperature": 0,
            "top_k": 250,
            "top_p": 0.9,
            "messages": [
                {
                    "role": "user",
                    "content": [{"text": prompt_data}]
                }
            ],
        })
        before = datetime.now()
        response = bedrock_runtime.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)
        latency = (datetime.now() - before)
        response_body = json.loads(response.get('body').read())
        response_text = response_body.get('content', [{}])[0].get('text')
        return response_text, latency
    else:
        print('Parameter model must be one of amazon titan, anthropic claude, or meta llama')
        return None, None

Now we are ready for running our examples with different models.

-----

## 1. Generate recommendations based on metadata

**Use Case:** A company wants to generate recommendations of flight destinations for their users based on some metadata, e.g. country, age-range, and interests.

**Task:** Text Generation

**Prompt Technique:** Zero-shot

In [5]:
prompt_data ="""
Human:
Generate a list of 10 recommended Melia Hotels for traveling considering the information in the <metadata></metadata> XML tags, and include a very brief description of each recommendation.
Answer in English.

<metadata>
Traveler going to Spain
Age range between 20-30
Interested on water sports and theme parks
Traveling during the summer
</metadata>

Assistant:
"""

In [6]:
# response, latency = call_bedrock('anthropic.claude-v2:1', prompt_data)
# print(response, "\n\n", "Inference time:", latency)

response, latency = call_bedrock('amazon.titan-text-express-v1', prompt_data)
print(response, "\n\n", "Inference time:", latency)

Here are 10 recommended Melia Hotels for traveling considering the information in the XML tags:

1. Melia Marbella Banus - Located in Marbella, Spain, this hotel offers a beachfront location, spa, and outdoor pool. It is ideal for travelers interested in water sports and theme parks.

2. Melia Villaitana - Located in Alicante, Spain, this hotel offers a golf course, spa, and outdoor pool. It is ideal for travelers interested in golf and outdoor activities.

3. Melia Caribe Beach Resort - Located in Punta Cana, Dominican Republic, this hotel offers a beachfront location, spa, and outdoor pool. It is ideal for travelers interested in water sports and beach activities.

4. Melia Nassau Beach - Located in Nassau, Bahamas, this hotel offers a beachfront location, spa, and outdoor pool. It is ideal for travelers interested in water sports and beach activities.

5. Melia Costa Del Sol - Located in Marbella, Spain, this hotel offers a beachfront location, spa, and outdoor pool. It is ideal for

------

## 2. Estimate capacity for airlines or hotel properties based on historical data

**Use Case:** A T&H company wants to estimate capacity/occupancy levels they could have for the next days based on historical information and flights/hotels metadata.

**Task:** Complex Reasoning

**Prompt Technique:** Chain-of-Thoughts (CoT)

In [7]:
prompt_data ="""
Human: Last week, the number of guests at 3 Melia hotels was as follows:
- Monday: Paris 650, New York 320, Singapore 415
- Tuesday: Paris 640, New York 330, Singapore 410
- Wednesday: Paris 630, New York 340, Singapore 425

Question: How many guests can we expect next Friday at the Paris hotel?
Answer: Based on the numbers given and without any further information, there is a daily decrease of 10 guests for the Paris hotel.
If we assume that this trend will continue for the next few days, we can expect 620 guests for the next day, which is Thursday, and
therefore, 610 guests for Friday.

Question: How many guests can we expect on Saturday at each of the hotels?
Give a step-by-step reasoning and recommendations for increasing the number of guests at the hotels.
Assistant:
"""

In [8]:
response, latency = call_bedrock('amazon.titan-text-express-v1', prompt_data)
print(response, "\n\n", "Inference time:", latency)

Step 1: Calculate the total number of guests for the week:
Total number of guests = Monday guests + Tuesday guests + Wednesday guests
Total number of guests = 650 + 640 + 630
Total number of guests = 1,920

Step 2: Calculate the number of guests for each day of the week:
Number of guests on Monday = 650
Number of guests on Tuesday = 640
Number of guests on Wednesday = 630
Number of guests on Thursday = 620
Number of guests on Friday = 610
Number of guests on Saturday = 610

Step 3: Increase the number of guests at the Paris hotel:
To increase the number of guests at the Paris hotel, we can consider the following strategies:

Increase marketing efforts: Promote the hotel through various marketing channels, such as social media, email marketing, and travel websites. Offer special promotions, discounts, or packages to attract more guests.

Offer additional amenities: Provide additional amenities such as a pool, spa, or fitness center to attract more guests.

Improve customer service: Ensu

------

## 3. Create a question answering assistant for customer service

**Use Case:** A company wants to create a bot capable of answering questions about the services available, based on the internal information for these.

**Task:** Question Answering with Dialogue Asistant (no memory)

**Prompt Technique:** Few-shot

In [9]:
prompt_data ="""
Context: An airline services available for purchase are as follows
1. Seat upgrades, available from 20 Euros, to classes Economy Plus and Business, on weekdays' flights
2. Meals, available for payment in-flight, mediterranean menu, on all days' flights
3. Fast boarding, from 30 Euros, available for premier customers, on all days' flights

Instruction: Answer any questions about the services available in a friendly manner. If you don't know the answer just say 'Apologies, but I don't have the answer for that. Please contact our team by phone.'

Assistant: Welcome to Airline Services, how can I help you?
Human: Hi, I would like to know what are the services available please.
Assistant: Of course, right now we have the seat upgrades, the meals, and the fast boarding.
Human: Thank you. I would like to know details for those please.
Assistant:
"""

In [10]:
response, latency = call_bedrock('arn:aws:bedrock:us-east-1:439319960869:inference-profile/us.meta.llama3-1-8b-instruct-v1:0', prompt_data)
print(response, "\n\n", "Inference time:", latency)



Our seat upgrades are available for purchase on weekdays, and they can be upgraded to either Economy Plus or Business class. The price for the upgrade is 20 Euros. 

Our meals are available for purchase on all days' flights, and we have a delicious Mediterranean menu for you to choose from. You can pay for them in-flight.

Lastly, our fast boarding service is available for our premier customers, and it costs 30 Euros. It's available on all days' flights. 

 Inference time: 0:00:00.583767


------

## 4. Generate content summary based on transcriptions from media files

**Use Case:** A company needs to generate summaries of the audio transcription for audio and video files for customer service, to be sent to their operations quality team. They also want to classify this content according to specific categories.

**Task:** Text Summarization & Text Classification

**Prompt Technique:** Zero-shot

#### (Pre-requisite)

First, we will need to transcribe the media files. You can e.g. use Amazon Transcribe for this task following examples like this: https://docs.aws.amazon.com/code-library/latest/ug/transcribe_example_transcribe_StartTranscriptionJob_section.html

For our sample we will start from an interview transcription in the file "interview.txt".

In [11]:
f = open("interview.txt", "r").read()
print(f)

Presenter: (00:00)
Thank you for doing This Christmas Will Be Different. I appreciate you doing that. We hung out all day together.

Actor Smith: (00:04)
How fun was that?

Presenter: (00:05)
It was the greatest. We always…

Actor Smith: (00:07)
Two take maximum. Two take maximum.

Presenter: (00:09)
We nailed it, we have some good takes. That was so much fun. Happy holidays to you and your family. I’m happy to see you in person. Thanks for making the trip and being here.

Actor Smith: (00:18)
Absolutely.

Presenter: (00:19)
Do you have like family traditions? Do you have holiday plans? What do you have coming up?

Actor Smith: (00:24)
This year, we’re going to get all the Smith family. We’re also getting Camila’s family. So when we do that and we have that multicultural Christmas, Christmas lasts a long time at our place because they celebrate on the 24th. Midnight you exchange presents. All right. So you have the big sit down dinner, you get the china out for the first time all year.

In [12]:
prompt_data =f"""
Human:
Generate a summary of the transcription in the <transcription></transcription> XML tags below.
Then, classify the mood of the participants according to the closest category within 'fully satisfied', 'satisfied', 'unsatisfied', 'fully unsatisfied', or 'neutral'.

<transcription>
{f}
</transcription>

Assistant:
"""

In [13]:
response, latency = call_bedrock('arn:aws:bedrock:us-east-1:439319960869:inference-profile/us.meta.llama3-1-8b-instruct-v1:0', prompt_data)
print(response, "\n\n", "Inference time:", latency)



**Summary:**

The transcription is a conversation between a presenter and Actor Smith, discussing various topics including their personal experiences, holiday traditions, and charitable work. The conversation starts with Actor Smith sharing his fun experience filming a Christmas special with the presenter, and then delves into his family's holiday traditions, which involve a large and multicultural celebration. The presenter and Actor Smith also discuss his book "Green Lights" and its success, as well as his upcoming journal companion. The conversation also touches on Actor Smith's decision not to run for governor of Texas, and his philanthropic work, including his foundation's efforts to provide disaster relief after a winter storm in Texas.

**Mood Classification:**

Based on the conversation, I would classify the mood of the participants as follows:

* Presenter: Satisfied (The presenter seems to be enjoying the conversation and is clearly enthusiastic about Actor Smith's work and

In [14]:
response, latency = call_bedrock('arn:aws:bedrock:us-east-1:439319960869:inference-profile/us.meta.llama3-1-8b-instruct-v1:0', prompt_data)
print(response, "\n\n", "Inference time:", latency)



**Summary:**

The transcription is a conversation between a presenter and Actor Smith, discussing various topics including their personal experiences, holiday traditions, and charitable work. The conversation starts with Actor Smith sharing his fun experience filming a Christmas special with the presenter, and then delves into his family's holiday traditions, which involve a large and multicultural celebration. The presenter and Actor Smith also discuss his book "Green Lights" and its success, as well as his upcoming journal companion. The conversation also touches on Actor Smith's decision not to run for governor of Texas, and his philanthropic work, including his foundation's efforts to provide disaster relief after a winter storm in Texas.

**Mood Classification:**

Based on the conversation, I would classify the mood of the participants as follows:

* Presenter: Satisfied (The presenter seems to be enjoying the conversation and is clearly enthusiastic about Actor Smith's work and

------

## 5. Create splash pages describing upcoming promotions

**Use Case:** A company wants to create HTML pages quickly and easily for their upcoming promotions.

**Task:** Code Generation

**Prompt Technique:** Zero-shot

In [15]:
prompt_data ="""
There is an upcoming promotion presented by the Melia Salou Hotel.
The promotion is targeting young audience in the age range between 18 and 40.
The promotion consists of a 20% discount when booking rooms online.
There will be additional fees for upgrades and bookings can be done trought this same portal.
The promotion is part of the Summer Discounts of the company.
The promotion is available from June 28th to August 31st.

Based the this information, generate the HTML code for an attractive splash page for this promotion.
Include catchy phrases and invite customers to sign-up for the Melia Hotel's loyalty program.
Have the splash page use blue fonts and black background, according to the hotel's branding.
"""

In [16]:
response, latency = call_bedrock('arn:aws:bedrock:us-east-1:439319960869:inference-profile/us.meta.llama3-1-8b-instruct-v1:0', prompt_data)
#print(response, "\n\n", "Inference time:", latency)
from IPython.display import display, HTML
display(HTML(response))

In [17]:
response, latency = call_bedrock('arn:aws:bedrock:us-east-1:439319960869:inference-profile/us.meta.llama3-1-8b-instruct-v1:0', prompt_data)
#print(response, "\n\n", "Inference time:", latency)
from IPython.display import display, HTML
display(HTML(response))

------